
### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.1-8b-instant")

response = model.invoke("Write me an easy on ai.")
response.content

"Here's a simple poem about AI:\n\nArtificial Intelligence, so bright,\nHelping us with tasks, day and night.\nFrom chatbots to robots, it's a technological sight,\nMaking our lives easier, with all its might.\n\nIt learns and adapts, with each passing test,\nImproving its skills, and doing its best.\nFrom language translation, to medical analysis too,\nAI is a tool, that's helping us get through.\n\nSo let's appreciate, this technological aid,\nThat's making our lives, a little less frayed.\nAI is a gift, that's here to stay,\nHelping us innovate, in a brighter way."

In [2]:
## Tools

from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""

    # In a real implementation, you would call a weather API here.
    return f"The current weather in {location} is sunny with a high of 25°C."

model_with_tools = model.bind_tools([get_weather])


In [4]:
response = model_with_tools.invoke("What's the weather like in Mumbai?")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'qxxntpq7b', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 224, 'total_tokens': 240, 'completion_time': 0.027219982, 'completion_tokens_details': None, 'prompt_time': 0.112166563, 'prompt_tokens_details': None, 'queue_time': 0.045080714, 'total_time': 0.139386545}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d70bf-e255-7971-b4cb-47047494321f-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'qxxntpq7b', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 224, 'output_tokens': 16, 'total_tokens': 240}


In [5]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Mumbai'},
  'id': 'qxxntpq7b',
  'type': 'tool_call'}]

### Tool execution loop

In [8]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Mumbai?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."


I'm sorry but I am unable to provide real-time weather updates.
